In [1]:
# !pip install pip install umap-learn
# !pip install pytorch-lightning
import tarfile
from PIL import Image
from glob import glob
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd
import umap
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping

In [2]:
import matplotlib
font = {'weight' : 'regular',
        'size'   : 22}

matplotlib.rc('font', **font)

In [3]:
!curl -O https://gist.githubusercontent.com/jszym/479db2af32411b64249bfb1bff43a95e/raw/1bef32b78e6ddd414beebe6e57cf3b0a0fac44ea/dictlogger.py
from dictlogger import DictLogger

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100   1610 100   1610   0      0  28890      0                              0
100   1610 100   1610   0      0  28698      0                              0
100   1610 100   1610   0      0  28544      0                              0


# ECSE552 - W25 - Assignment 2 - Part 2 (Total 20 points)

## Conditional Variational Autoencoders

### Note

***Clearly comment each of your steps. Failure to do so would result in loss of points.***
**Additionally explain each line in your code**

### Introduction

Conditional Variational Autoencoders (CVAEs) are an extension of Variational Autoencoders (VAEs).

As we explored in tutorial eight, VAEs are comprised of two main parts: an encoder and a decoder.

The encoder approximates the function $Q(z|X)$ and the decoder approximates the function $P(X|z)$.

As a reminder, here $X$ is the input data and $z$ is a latent vector.

For more details, you might consider this [Tutorial on Variational Autoencoders](https://arxiv.org/abs/1606.05908) by Carl Doersch.

The above formulation, however, does not allow us to generate specific images. For example, given a model trained on the MNIST digits, it's difficult to fully explore the latent space for a given digit.

We can solve this by conditioning $P$ by not only the latent vector $z$, but some value $c$ that can encode information, (_e.g._: which MNIST digit we're generating).

This would mean our decoder would got from $P(X|z)$ in the VAE case to $P(X|z,c)$ in the CVAE case.

### Practical Implementation

Practically, this means that a "conditional" vector $c$ is passed to the decoder. This is commonly a class label for the sample that is one-hot encoded, but it can be anything you wish to marginalise the model by.

Below is a very high-level illustration of both a Variational Autoencoder (VAE) and a Conditional Variational Autoencoder (CVAE). Components of the CVAE that are not present in the VAE are demarked with a yellow star.

The CVAE is identical to the VAE, but for the concatenation of the vector $c$ to the vector $z$ before being inputted into the decoder. The loss function does not change in any way.

In this diagram the dimensions are noted for all the inputs and outputs of the Encoder and Decoder.

The dimension variables are as follows:

* $m$ is the batch size
* $w$ is the width of the image inputted into our encoder
* $h$ is the height of the image inputted into our encoder
* $n$ is the number of latent variables
* $q$ is the width of the conditional vector. If $c$ represents the class of the image, then $q$ is often equal to the number of classes.

<img src="https://ecse552.b-cdn.net/W24/H3/vae_cvae.webp" alt="VAE vs CVAE" width="50%"/>

Let's look closer at the concatenation of $z$ and $c$, as well as the representation of $c$. Consider the case of a CVAE where the samples are MNIST digits and $c$ is the one-hot encoded class of the digit. In this example, the number of latent variables ($n$ in the diagram above) is 3. Below is an illustration of what the concatenation might look like.

![Concatenating z with c](https://ecse552.b-cdn.net/W24/H3/conditional_z.webp)

The vectors that are the result of concatenating $z$ and $c$ (let's call it $z_c$) then become the input to the decoder.

### Q1.1 - Implementing an MNIST CVAE (10 points)

Your task is to extend the MNIST VAE from Tutorial 8 to a CVAE.

Specifically, you must:

1. Correctly define a conditional vector $c$ in the training loop.
2. Correctly concatenate it to $z$ to form $z_c$.
3. Input $z_c$ to the Decoder.
4. Make any further nescessary modifications to the code.

You are provided with the complete code for the MNIST VAE presented in Tutorial 8, including data loaders, encoders, decoders, and training/validation loops.

The number of latent variables for this excercise ($n$) is set to 10.

**Hint**: While you are permitted to make changes to the Encoder and Decoder class, it is not nescessary to correctly answer this question.

**Another Hint**: You can correctly answer this question by modifying/adding no more than six lines.

In [4]:
class MNISTDataset(Dataset):

    def __init__(self, csv_path):

        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        return torch.tensor(self.df.iloc[idx][1:])/255, self.df.iloc[idx][0]

In [5]:
batch_size = 100

dataset = MNISTDataset('./sample_data/mnist_train_small.csv')

num_test = len(dataset) // 10
num_train = len(dataset) - num_test
dataset_train, dataset_test = random_split(dataset, [num_train, num_test])

dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(dataset_test, batch_size=batch_size, shuffle=True)

In [6]:
class Encoder(nn.Module):
    def __init__(self, in_dim, latent_dim):
        super(Encoder, self).__init__()

        self.layer0 = nn.Linear(in_dim, 1024)
        self.layer1 = nn.Linear(512, 768)
        self.layer2 = nn.Linear(768, 1024)
        self.layer3 = nn.Linear(512, 256)
        self.layer4 = nn.Linear(128, latent_dim)

        self.pool = nn.MaxPool1d(2)

        self.dropout = nn.Dropout(0.3)

        self.activation = nn.LeakyReLU(0.01)

    def forward(self, x):

        x = self.layer0(x)
        x = self.activation(x)

        x = self.dropout(x)

        x = x.reshape(x.shape[0], 1, x.shape[1])
        x = self.pool(x)
        x = x.reshape(x.shape[0], x.shape[2])

        x = self.layer1(x)
        x = self.activation(x)

        x = self.dropout(x)

        x = self.layer2(x)
        x = self.activation(x)

        x = self.dropout(x)

        x = x.reshape(x.shape[0], 1, x.shape[1])
        x = self.pool(x)
        x = x.reshape(x.shape[0], x.shape[2])

        x = self.layer3(x)
        x = self.activation(x)

        x = self.dropout(x)

        x = x.reshape(x.shape[0], 1, x.shape[1])
        x = self.pool(x)
        x = x.reshape(x.shape[0], x.shape[2])

        x = self.layer4(x)
        x = self.activation(x)

        return x

In [7]:
class Decoder(nn.Module):
    def __init__(self, latent_dim, out_dim):
        super(Decoder, self).__init__()

        self.layer4 = nn.Linear(784, out_dim)
        self.layer3 = nn.Linear(768, 784)
        self.layer2 = nn.Linear(1024, 768)
        self.layer1 = nn.Linear(256, 1024)
        self.layer0 = nn.Linear(latent_dim, 256)

        self.dropout = nn.Dropout(0.3)

        self.activation = nn.LeakyReLU(0.01)

    def forward(self, x):

        x = self.layer0(x)
        x = self.activation(x)

        x = self.dropout(x)

        x = self.layer1(x)
        x = self.activation(x)

        x = self.dropout(x)

        x = self.layer2(x)
        x = self.activation(x)

        x = self.dropout(x)

        x = self.layer3(x)
        x = self.activation(x)

        x = self.dropout(x)

        x = self.layer4(x)
        x = torch.sigmoid(x)

        return x

In [8]:
class MNIST_VAE(pl.LightningModule):
    def __init__(self, in_dim, out_dim, latent_dim):
        super(MNIST_VAE, self).__init__()

        self.latent_dim = latent_dim

        self.encoder = Encoder(in_dim, latent_dim*2)
        self.decoder = Decoder(latent_dim+10, out_dim)  # accept 20 input features in total, 10 from z and 10 from c
        self.recon_criterion = nn.BCELoss(reduction='sum')

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5*log_var)
        eps = torch.randn_like(std)
        return eps.mul(std).add_(mu)

    def loss(self, mu, logvar, pred, target):

        kld_loss = torch.sum(-0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp(), dim = 1), dim = 0)
        recons_loss = self.recon_criterion(pred, target)

        loss = recons_loss + kld_loss

        return kld_loss, recons_loss, loss

    def forward(self, x, c):    # Receive the input image x and its conditional vector c. The model will now receive the digit's class information alongside the image

        parameters= self.encoder(x)

        mu = parameters[:, :self.latent_dim]
        logvar = parameters[:, self.latent_dim:]

        z = self.reparameterize(mu, logvar)

        x_hat = self.decoder(torch.cat((z, c), dim=1))  # Concatenate the latent vector and class condition before decoding

        return mu, logvar, z, x_hat

    def training_step(self, batch, batch_idx):

        x, y = batch
        c = F.one_hot(y, num_classes=10).float()  # Convert the digit labels into 10-dimensional one-hot vectors

        mu, logvar, z, x_hat = self(x, c)  # Reconstruct each image using its correct class condition

        kld_loss, recons_loss, loss = self.loss(mu, logvar, x_hat, x)

        self.log('training_loss', loss, on_step=False, on_epoch=True)
        self.log('training_kld_loss', kld_loss, on_step=False, on_epoch=True)
        self.log('training_recons_loss', recons_loss, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):

        x, y = batch

        mu, logvar, z, x_hat = self(x, F.one_hot(y, num_classes=10).float())  # Reconstruct validation images using their correct labels

        kld_loss, recons_loss, loss = self.loss(mu, logvar, x_hat, x)

        self.log('val_loss', loss, on_step=False, on_epoch=True)
        self.log('val_kld_loss', kld_loss, on_step=False, on_epoch=True)
        self.log('val_recons_loss', recons_loss, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.01, steps_per_epoch=len(dataloader_train), epochs=20)
        return [optimizer], [scheduler]

In [9]:
model = MNIST_VAE(784, 784, 10)

In [ ]:
logger = DictLogger()
early_stopping = EarlyStopping('val_loss', verbose=True, min_delta=0.0001,
                               patience=3)
# trainer = pl.Trainer(gpus=1, logger=logger, callbacks=[early_stopping],
#                      max_epochs=40)

# Create the trainer, automatically selecting GPU if available or CPU otherwise.
trainer = pl.Trainer(accelerator="auto", devices=1, logger=logger, callbacks=[early_stopping], max_epochs=40)


trainer.fit(model, dataloader_train, dataloader_test)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name            | Type    | Params | Mode  | FLOPs
------------------------------------------------------------
0 | encoder         | Encoder | 2.1 M  | train | 0    
1 | decoder         | Decoder | 2.3 M  | train | 0    
2 | recon_criterion | BCELoss | 0      | train | 0    
------------------------------------------------------------
4.4 M     Trainable params
0         Non-trainable params
4.4 M     Total params
17.573    Total estimated model params size (MB)
18        Modules

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\grace\OneDrive\Bureau\McGill\ECSE 552\552-Assignment-2\.venv\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\grace\OneDrive\Bureau\McGill\ECSE 552\552-Assignment-2\.venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:485: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
c:\Users\grace\OneDrive\Bureau\McGill\ECSE 552\552-Assignment-2\.venv\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
C:\Users\grace\AppData\Local\Temp\ipykernel_24184\3284235120.py:12: FutureWarning: Series.__getitem__ treating keys as po

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved. New best score: 20343.736


... (2 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved by 1106.775 >= min_delta = 0.0001. New best score: 17393.461


Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved by 643.607 >= min_delta = 0.0001. New best score: 16749.854


In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(logger.metrics['training_loss'], label='Training', lw=3)
plt.plot(logger.metrics['val_loss'], label='Val', lw=3)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(logger.metrics['training_kld_loss'], label='KLD', lw=3)
plt.plot(logger.metrics['training_recons_loss'], label='Recon', lw=3)
plt.legend()
plt.show()

### Q1.2 Demonstrate Reconstruction (5 points)

In the same way as was demonstrated in Tutorial 8, the ability of your model to reconstruct the first two digits from the first batch provided by `dataloader_test`.

Specifically, pass the first two values of `x` and `y` from `dataloader_test` to `model` and reconstruct these images. Plot them as illustrated below:

<img src=https://ecse552.b-cdn.net/W24/H3/q1.2.webp alt="Reconstructed MNIST digits" width="35%"/>

**Note**: The batches are randomised so you won't nescessarily get the same samples shown above.

In [ ]:
# Switch the trained CVAE into evaluation mode to disable dropout
model.eval()

# Retrieve the first batch of images (x) and digit labels (y) from the test DataLoader
x, y = next(iter(dataloader_test))

# Select the first two flattened images and move them to the model's device
x = x[:2].float().to(model.device)

# Select the corresponding digit labels and place them on the same device
y = y[:2].long().to(model.device)

# Convert the integer digit labels into 10-dimensional one-hot conditional vectors
c = F.one_hot(y, num_classes=10).float()

# Disable gradient tracking because no parameter updates are needed during reconstruction
with torch.no_grad():
    # Pass the first two images and their correct digit conditions to the trained CVAE
    mu, logvar, z, x_hat = model(x, c)

# Create a 2-by-2 figure with one row per image and one column per image type
fig, ax = plt.subplots(2, 2, figsize=(7, 7))

# Loop over the two samples to display each original and reconstructed image
for i in range(2):
    # Display the original image by reshaping its 784 pixels into a 28-by-28 grid
    ax[i, 0].imshow(x[i].reshape(28, 28).cpu().numpy(), cmap='Greys_r')
    # Identify the ground-truth class of the original image.
    ax[i, 0].set_title(f'Original (label {y[i].item()})')
    # Hide coordinate axes, which are not meaningful for MNIST images
    ax[i, 0].axis('off')
    # Display the corresponding CVAE reconstruction in the right-hand column
    ax[i, 1].imshow(x_hat[i].reshape(28, 28).cpu().numpy(), cmap='Greys_r')
    # Identify the digit label used to condition the reconstruction.
    ax[i, 1].set_title(f'Reconstructed (label {y[i].item()})')
    # Hide the reconstruction image's coordinate axes
    ax[i, 1].axis('off')

# Arrange the subplot titles and images neatly without overlaps
plt.tight_layout()

# Render the two original-versus-reconstructed pairs
plt.show()

### Q1.3 Demonstrate Conditional Reconstruction (5 points)

Similar to Q1.2, reconstruct digits, but instead of using the correct $c$ vector, change the class label to a different one.

For example, the value of `y` for the two samples below are 3 and 6, but when reconstructing, the $c$ vector `[0,0,1,0,0,0,0,0,0,0]` was supplied.

<img src="https://ecse552.b-cdn.net/W24/H3/q1.3.png" alt="Conditioned Reconstruction" width="35%"/>

#### Code explanation for Q1.3 - change the class condition

This code chooses the first two images from the first test batch, then computes reconstructions using `correct` and `intentionally incorrect` one-hot class vectors. `y_changed = (y + 1) % 10` guarantees the new class is different for every digit (including 9, which changes to 0).

**Why reuse `z`?** The model's `forward` method randomly samples its latent vector through reparameterization. We compute it only once, then pass that exact same `z` to the decoder together with a new `c`. This controls the randomness: the condition is the only input to the decoder that changes between the correct and incorrect reconstructions.

The three columns show the original image, the normal reconstruction with the real label, and a reconstruction with the changed label. The conditional image may move toward the requested digit, but this is not guaranteed if the CVAE has learned to ignore or weakly use class information. Results depend on training.

In [ ]:
# Put the already-trained CVAE in evaluation mode so dropout is disabled
model.eval()

# Retrieve the first batch of MNIST images and their original digit labels
batch_x, batch_y = next(iter(dataloader_test))

# Keep only the first two flattened images and move them to the model's device
x = batch_x[:2].float().to(model.device)

# Keep the matching integer labels and move them to the model's device
y = batch_y[:2].long().to(model.device)

# Encode the real labels as 10-dimensional one-hot vectors for the correct reconstruction
c_correct = F.one_hot(y, num_classes=10).float()

# Change every label to the next digit (9 wraps to 0), so the new class is always different
y_changed = (y + 1) % 10

# One-hot encode the deliberately incorrect class labels for conditional reconstruction
c_changed = F.one_hot(y_changed, num_classes=10).float()

# Turn off gradient tracking because we are comparing reconstructions, not training
with torch.no_grad():
    # Encode and reconstruct the original images using their real class labels
    mu, logvar, z, x_hat_correct = model(x, c_correct)

    # Keep the same latent vectors z but replace the conditions to isolate the effect of c
    x_hat_changed = model.decoder(torch.cat((z, c_changed), dim=1))

# Make two rows (one per image) and three columns (original, correct, changed condition)
fig, axes = plt.subplots(2, 3, figsize=(10, 7))

# Repeat the visualization for both selected test digits
for i in range(2):
    # Display the original 28-by-28 MNIST image
    axes[i, 0].imshow(x[i].reshape(28, 28).cpu().numpy(), cmap="Greys_r")

    # Identify the digit that the image actually represents
    axes[i, 0].set_title(f"Original: {y[i].item()}")

    # Remove pixel-coordinate ticks around the original image
    axes[i, 0].axis("off")

    # Display the reconstruction obtained using the correct one-hot class vector
    axes[i, 1].imshow(x_hat_correct[i].reshape(28, 28).cpu().numpy(), cmap="Greys_r")

    # Label this image using the true digit class supplied to the decoder
    axes[i, 1].set_title(f"Correct condition: {y[i].item()}")

    # Hide unnecessary pixel-coordinate ticks around the correct reconstruction
    axes[i, 1].axis("off")

    # Display the reconstruction produced with a different class condition
    axes[i, 2].imshow(x_hat_changed[i].reshape(28, 28).cpu().numpy(), cmap="Greys_r")

    # Show the incorrect class label that was intentionally passed to the decoder
    axes[i, 2].set_title(f"Changed condition: {y_changed[i].item()}")

    # Hide unnecessary pixel-coordinate ticks around the conditional reconstruction
    axes[i, 2].axis("off")

# Automatically adjust spacing so titles and images do not overlap
plt.tight_layout()

# Render the comparison of original, normal, and conditionally reconstructed digits
plt.show()
